# Extrac Public Firm Financial Data

In [1]:
import os
import zipfile
from io import BytesIO

import requests
import pandas as pd

# SEC requires a descriptive User-Agent with a contact address.
HEADERS = {"User-Agent": "Xiangtao M xiangtaom@gmail.com"}


def download_and_unzip(url: str, extract_to: str) -> None:
    """Download a ZIP from `url` and extract its contents into `extract_to`."""
    os.makedirs(extract_to, exist_ok=True)
    print(f"Downloading {url} ...")
    resp = requests.get(url, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    with zipfile.ZipFile(BytesIO(resp.content)) as zf:
        print(f"Extracting {zf.namelist()} to {extract_to} ...")
        zf.extractall(path=extract_to)
    print("Done.")


## download 2026Q1

In [2]:
quarter = "2026q1"
data_dir = f"../data/sec/{quarter}"   # notebook is in src/, so ../data
download_and_unzip(
    f"https://www.sec.gov/files/dera/data/financial-statement-data-sets/{quarter}.zip",
    extract_to=data_dir,
)
# -> data_dir now holds sub.txt, num.txt, pre.txt, tag.txt, readme.htm


Extracting ['sub.txt', 'pre.txt', 'num.txt', 'tag.txt', 'readme.htm'] to ../data/sec/2026q1 ...
Done.


In [3]:
firms = pd.DataFrame(
    [
        ("AAPL",  320193, "Apple"),
        ("MSFT",  789019, "Microsoft"),
        ("AMZN", 1018724, "Amazon"),
        ("NVDA", 1045810, "Nvidia"),
        ("GOOGL", 1652044, "Alphabet"),
        ("TSLA", 1318605, "Tesla"),
        ("META", 1326801, "Meta"),
    ],
    columns=["ticker", "cik", "company"],
)

# sub.txt = submissions. Tab-separated; keep_default_na=False so blank strings stay "".
sub = pd.read_csv(f"{data_dir}/sub.txt", sep="\t", low_memory=False, keep_default_na=False)

sub7 = sub[sub["cik"].isin(firms["cik"])].merge(firms, on="cik", how="left")
sub7[["ticker", "cik", "name", "form", "period", "fy", "fp", "filed", "adsh"]]


,ticker,cik,name,form,period,fy,fp,filed,adsh
0,MSFT,789019,MICROSOFT CORP,11-K,20251231,2025,FY,20260324,0001193125-26-120090
1,AAPL,320193,APPLE INC,10-Q,20251231,2026,Q1,20260130,0000320193-26-000006
2,AMZN,1018724,AMAZON COM INC,10-K,20251231,2025,FY,20260206,0001018724-26-000004
3,NVDA,1045810,NVIDIA CORP,10-K,20260131,2026,FY,20260225,0001045810-26-000021
4,MSFT,789019,MICROSOFT CORP,10-Q,20251231,2026,Q2,20260128,0001193125-26-027207
5,META,1326801,"META PLATFORMS, INC.",10-K,20251231,2025,FY,20260129,0001628280-26-003942
6,TSLA,1318605,"TESLA, INC.",10-K,20251231,2025,FY,20260129,0001628280-26-003952
7,GOOGL,1652044,ALPHABET INC.,10-K,20251231,2025,FY,20260205,0001652044-26-000018


In [4]:
# num.txt = numeric facts. adsh links a fact to its submission in sub.txt.
num = pd.read_csv(f"{data_dir}/num.txt", sep="\t", low_memory=False)

facts7 = num.merge(
    sub7[["adsh", "ticker", "cik", "company", "form", "period", "fy", "fp"]],
    on="adsh",
    how="inner",
)
print(f"{len(facts7):,} numeric facts for the 7 firms")

# Example: pull one tag across firms (tag, qtrs, uom, coreg all matter when filtering)
example = facts7[facts7["tag"] == "Assets"][
    ["ticker", "company", "tag", "ddate", "qtrs", "uom", "value"]
].sort_values(["company", "ddate"])
example.head(20)


3,098 numeric facts for the 7 firms


,ticker,company,tag,ddate,qtrs,uom,value
2100,GOOGL,Alphabet,Assets,20241231,0,USD,4.502560e+11
2258,GOOGL,Alphabet,Assets,20241231,0,USD,8.700000e+09
2205,GOOGL,Alphabet,Assets,20251231,0,USD,5.600000e+09
3052,GOOGL,Alphabet,Assets,20251231,0,USD,5.952810e+11
240,AMZN,Amazon,Assets,20231231,0,USD,1.085330e+11
291,AMZN,Amazon,Assets,20231231,0,USD,6.971800e+10
419,AMZN,Amazon,Assets,20231231,0,USD,5.278540e+11
550,AMZN,Amazon,Assets,20231231,0,USD,1.535740e+11
2699,AMZN,Amazon,Assets,20231231,0,USD,1.960290e+11
241,AMZN,Amazon,Assets,20241231,0,USD,6.948700e+10


## Quarterly profitability & performance variables

Build a tidy company panel of the line items most used to gauge profitability and
performance, then derive standard ratios.

**Extraction rules**
- *Consolidated only*: keep `coreg == ""` (no co-registrant) and `segments == ""`
  (no dimensional / segment breakdown), so we get company-level totals.
- *Flows vs. stocks*: income-statement items are flows measured over one quarter
  (`qtrs == 1`); balance-sheet items are point-in-time stocks (`qtrs == 0`).
- *Current period*: keep facts whose `ddate` equals the filing's `period`
  (drops the prior-year comparatives that filings also carry).
- *Tag coalescing*: firms tag the same concept differently
  (e.g. Alphabet/Nvidia use `Revenues`, the others use
  `RevenueFromContractWithCustomerExcludingAssessedTax`), so each concept maps to a
  priority list of candidate tags.

**Single-quarter caveat.** In the 2026Q1 test set, quarterly income-statement flows
(`qtrs == 1`) exist only for firms that filed a **10-Q** this quarter (Apple, Microsoft).
The 10-K filers report annual figures only, so their flow variables come back `NaN`;
balance-sheet stocks are available for all seven. See the closing note for how to
assemble a full quarterly panel across many quarters.

In [ ]:
# Consolidated, numeric-typed facts for the 7 firms.
# NOTE: num.txt is best read with keep_default_na=False so blank segments/coreg stay "".
# facts7 above was read without it, so blanks are NaN -> normalize with fillna("").
core = facts7.copy()
core["value"]  = pd.to_numeric(core["value"],  errors="coerce")
core["ddate"]  = pd.to_numeric(core["ddate"],  errors="coerce")
core["period"] = pd.to_numeric(core["period"], errors="coerce")
core = core[(core["coreg"].fillna("") == "") & (core["segments"].fillna("") == "")]
print(f"{len(core):,} consolidated facts for the 7 firms")

In [ ]:
import numpy as np

# concept -> (candidate tags in priority order, "flow" | "stock")
CONCEPTS = {
    "revenue":             (["RevenueFromContractWithCustomerExcludingAssessedTax", "Revenues"], "flow"),
    "cost_of_revenue":     (["CostOfGoodsAndServicesSold", "CostOfRevenue"], "flow"),
    "gross_profit":        (["GrossProfit"], "flow"),
    "operating_income":    (["OperatingIncomeLoss"], "flow"),
    "rd_expense":          (["ResearchAndDevelopmentExpense"], "flow"),
    "net_income":          (["NetIncomeLoss"], "flow"),
    "total_assets":        (["Assets"], "stock"),
    "stockholders_equity": (["StockholdersEquity"], "stock"),
}


def pick(df, tags, qtrs):
    """Current-period value per company for the first available tag (by priority)."""
    d = df[(df["qtrs"] == qtrs) & (df["tag"].isin(tags)) & (df["ddate"] == df["period"])].copy()
    d["prio"] = d["tag"].map({t: i for i, t in enumerate(tags)})
    d = d.sort_values(["company", "prio"]).drop_duplicates("company", keep="first")
    return d.set_index("company")["value"]


financials = pd.DataFrame(index=sorted(core["company"].unique()))
for concept, (tags, kind) in CONCEPTS.items():
    financials[concept] = pick(core, tags, 1 if kind == "flow" else 0)

# Gross profit isn't tagged by every firm -> fall back to revenue - cost of revenue.
financials["gross_profit"] = financials["gross_profit"].fillna(
    financials["revenue"] - financials["cost_of_revenue"]
)

# Attach reporting-period identifiers (ticker, fiscal year/period, period-end date).
ids = core.drop_duplicates("company").set_index("company")[["ticker", "fy", "fp", "period"]]
financials = ids.join(financials)
financials

In [ ]:
# Standard profitability & performance ratios.
# Margins use one quarter of flows. ROA/ROE/turnover mix a quarterly flow with a
# period-end stock, so they are *quarterly* (not annualized) figures.
ratios = pd.DataFrame(index=financials.index)
ratios["ticker"]             = financials["ticker"]
ratios["gross_margin"]       = financials["gross_profit"]     / financials["revenue"]
ratios["operating_margin"]   = financials["operating_income"] / financials["revenue"]
ratios["net_margin"]         = financials["net_income"]       / financials["revenue"]
ratios["rd_intensity"]       = financials["rd_expense"]       / financials["revenue"]
ratios["roa_qtr"]            = financials["net_income"]       / financials["total_assets"]
ratios["roe_qtr"]            = financials["net_income"]       / financials["stockholders_equity"]
ratios["asset_turnover_qtr"] = financials["revenue"]          / financials["total_assets"]
ratios.round(3)

### Extending to a full quarterly panel

This test used a single quarter, so quarterly flows appear only for the two 10-Q filers.
To build a complete quarterly time series for all seven firms:

1. Loop `download_and_unzip` over every quarter you need (e.g. 2020q1 … 2026q1) and
   `pd.concat` the `sub` / `num` tables (see `Populate_Data.py` and example 2).
2. Each firm's first three fiscal quarters come from its 10-Q filings (`qtrs == 1`).
3. The **fourth fiscal quarter** is never reported directly — derive it as
   annual (`qtrs == 4`, from the 10-K) minus the sum of the first three quarters.
4. Concatenate all quarters, drop duplicates on `(cik, tag, ddate, qtrs)` keeping the
   latest `filed` (restatements), then pivot to a `company x quarter` panel before
   computing the ratios above.